In [5]:

import os
import pandas as pd
import json
import csv
import re

# Input and output paths
input_folder = "project_folder/LLM/New_documents_raw_outputs"
output_dir = "project_folder/LLM/New_documents_processed"
os.makedirs(output_dir, exist_ok=True)

def process_single_csv(input_csv, output_dir):
    # Extract document_id from filename (e.g., claude_v1_49.csv -> 49)
    document_id = os.path.splitext(os.path.basename(input_csv))[0].split('_')[-1]
    rows = []
    with open(input_csv, newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            raw_output = row.get('raw_output', '')
            try:
                # Find the first '{' and the last '}' to extract the JSON substring
                start = raw_output.find('{')
                end = raw_output.rfind('}')
                if start != -1 and end != -1 and end > start:
                    raw_output_clean = raw_output[start:end+1]
                    # Remove trailing commas before closing braces/brackets
                    raw_output_clean = re.sub(r',(\s*[}\]])', r'\1', raw_output_clean)
                    # Remove newlines inside the JSON string
                    raw_output_clean = re.sub(r'\n', ' ', raw_output_clean)
                    # Remove double commas (,,) which can occur after removing trailing commas
                    raw_output_clean = re.sub(r',\s*,', ',', raw_output_clean)
                    # Remove any comma directly before a closing square bracket
                    raw_output_clean = re.sub(r',\s*\]', ']', raw_output_clean)
                    # Remove any comma directly before a closing curly brace
                    raw_output_clean = re.sub(r',\s*\}', '}', raw_output_clean)
                    # Truncate at the last complete constraint (last closing curly brace before the file ends)
                    last_constraint_end = raw_output_clean.rfind('}')
                    if last_constraint_end != -1:
                        truncated = raw_output_clean[:last_constraint_end+1]
                        # Try to close any open brackets for arrays/objects
                        open_braces = truncated.count('{')
                        close_braces = truncated.count('}')
                        open_brackets = truncated.count('[')
                        close_brackets = truncated.count(']')
                        truncated += ']' * (open_brackets - close_brackets)
                        truncated += '}' * (open_braces - close_braces)
                        raw_output_clean = truncated
                    raw = json.loads(raw_output_clean)
                    # Build component_id -> component_name mapping
                    comp_map = {}
                    for comp in raw.get('project_components', []):
                        comp_map[comp.get('component_id')] = comp.get('component_name')
                    # Use document_id from filename, model_type/model_name from row
                    model_type = row.get('model_type', '')
                    model_name = row.get('model_name', '')
                    # Process constraints
                    for constraint in raw.get('project_constraints', []):
                        constraint_row = constraint.copy()
                        # Replace linked_component_id with component_name
                        comp_id = constraint_row.pop('linked_component_id', None)
                        constraint_row['component_name'] = comp_map.get(comp_id, comp_id)
                        # Add document/model info
                        constraint_row['document_id'] = document_id
                        constraint_row['model_type'] = model_type
                        constraint_row['model_name'] = model_name
                        # Reorder columns: model_type, model_name, document_id, then the rest
                        ordered_row = {
                            'model_type': model_type,
                            'model_name': model_name,
                            'document_id': document_id
                        }
                        ordered_row.update(constraint_row)
                        rows.append(ordered_row)
                else:
                    print(f"No valid JSON object found in raw_output for {input_csv}.")
            except Exception as e:
                print(f"Error processing row in {input_csv}: {e}")
    # Output DataFrame
    if rows:
        out_df = pd.DataFrame(rows)
        cols = ['model_type', 'model_name', 'document_id'] + [c for c in out_df.columns if c not in ['model_type', 'model_name', 'document_id']]
        out_df = out_df[cols]
        output_csv = os.path.join(
            output_dir,
            os.path.splitext(os.path.basename(input_csv))[0] + "_processed.csv"
        )
        out_df.to_csv(output_csv, index=False)
        print(f"Processed CSV saved to: {output_csv}")
        return out_df
    else:
        print(f"No constraints found in {input_csv}.")
        return None

# Process all CSVs in the input folder and combine
all_rows = []
for fname in sorted(os.listdir(input_folder)):
    if fname.endswith(".csv"):
        input_csv = os.path.join(input_folder, fname)
        df = process_single_csv(input_csv, output_dir)
        if df is not None and not df.empty:
            all_rows.append(df)

# Combine all processed CSVs into one big CSV
if all_rows:
    combined_df = pd.concat(all_rows, ignore_index=True)
    combined_csv = os.path.join(output_dir, "all_processed_constraints.csv")
    combined_df.to_csv(combined_csv, index=False)
    print(f"Combined CSV saved to: {combined_csv}")
else:
    print("No processed constraints to combine.")

Processed CSV saved to: project_folder/LLM/New_documents_processed/claude_v1_1_processed.csv
Processed CSV saved to: project_folder/LLM/New_documents_processed/claude_v1_10_processed.csv
Processed CSV saved to: project_folder/LLM/New_documents_processed/claude_v1_11_processed.csv
Processed CSV saved to: project_folder/LLM/New_documents_processed/claude_v1_12_processed.csv
Processed CSV saved to: project_folder/LLM/New_documents_processed/claude_v1_13_processed.csv
Processed CSV saved to: project_folder/LLM/New_documents_processed/claude_v1_14_processed.csv
Processed CSV saved to: project_folder/LLM/New_documents_processed/claude_v1_15_processed.csv
Processed CSV saved to: project_folder/LLM/New_documents_processed/claude_v1_16_processed.csv
Processed CSV saved to: project_folder/LLM/New_documents_processed/claude_v1_17_processed.csv
Processed CSV saved to: project_folder/LLM/New_documents_processed/claude_v1_19_processed.csv
Processed CSV saved to: project_folder/LLM/New_documents_proc